# Data preprocessing

This notebook contains executed work for Phase 1 Data Cleaning, Phase 2 Data Selection and Phase 3 Data Transformation. Descriptive analytics, visualizations and mining/model implementations belong to later phases.

## Phase 1 Data Cleaning

We clean all 37 original Kaggle tables while preserving their native observation units. The Phase 1 section executes the reusable cleaning pipeline and displays its quality evidence.

### Cleaning rules

We validate CSV widths and explicitly restore the missing religion header. We standardize clear geography/category aliases and use nullable numeric types. Unknown counts remain missing unless two independent within-record equations agree. Exact duplicate crime records are removed; conflicting crime keys are quarantined. Training and goods records have an under-specified grain, so repeated keys are preserved and flagged. Totals, subtotals and detail records are distinguished in a companion quality table. Outliers are flagged with contextual 1.5 IQR fences and retained; zero-IQR groups are not automatically flagged.

In [1]:
from pathlib import Path
import sys

# find the repository from either the root or notebooks directory
ROOT = Path.cwd().resolve()
if not (ROOT / "question.txt").exists() and (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "src/preprocessing/cleaning.py").is_file()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json
import pandas as pd
from IPython.display import display
from src.preprocessing.cleaning import run_cleaning, sha256

In [2]:
# run cleaning and verify that raw inputs match the saved manifest
summary = run_cleaning(ROOT)
print(json.dumps(summary, indent=2))
assert summary["raw_hashes_unchanged"]

{
  "files": 37,
  "rows_before": 159032,
  "rows_after": 159019,
  "duplicates_removed": 3,
  "quarantined_rows": 10,
  "missing_cells_before": 2232,
  "missing_cells_after": 2230,
  "counts_recovered": 2,
  "outlier_cells_flagged": 56018,
  "raw_hashes_unchanged": true
}


In [3]:
reports = ROOT / "results/tables/cleaning"
quality = pd.read_csv(reports / "quality_summary.csv")
display(quality[["file", "rows_before", "rows_after", "duplicates_removed",
                 "quarantined_rows", "missing_cells_before", "missing_cells_after"]])

,file,rows_before,rows_after,duplicates_removed,quarantined_rows,missing_cells_before,missing_cells_after
0,Age_group.csv,7152,7152,0,0,1,1
1,Caste.csv,3568,3568,0,0,0,0
2,Death_sentence.csv,446,446,0,0,0,0
3,Domicile.csv,2676,2676,0,0,0,0
4,Education.csv,5346,5346,0,0,0,0
5,Education_facilities.csv,445,445,0,0,0,0
6,IPC_crime_inmates_convicted.csv,9391,9385,2,4,1,0
7,IPC_crime_inmates_under_trial.csv,9390,9385,1,4,0,0
8,Inmates_death.csv,4023,4023,0,0,0,0
9,Inmates_escapee.csv,1788,1788,0,0,0,0


The missing-value baseline is measured after validated schema alignment but before value cleaning. This avoids treating the shifted Religion.csv header as if it were a valid schema. Missing contact information is kept unknown. Removing duplicates or quarantining a row is accounted for separately from repairing cells.

In [4]:
columns = pd.read_csv(reports / "quality_columns.csv")
display(columns.loc[columns["missing"].gt(0),
                    ["file", "stage", "column", "dtype", "missing"]])
display(pd.read_csv(reports / "cell_actions.csv"))

,file,stage,column,dtype,missing
6,Age_group.csv,before,age_16_18,string,1
16,Age_group.csv,after,age_16_18,Int64,1
99,IPC_crime_inmates_convicted.csv,before,Male 16-18 years,string,1
397,Other Prison data/SLL_Crime_headwise_distribut...,before,Female 18-30 years,string,1
501,Other Prison data/Value_of_goods_produced_by_i...,before,Gross Value of Sale Proceeds / Earning (In Rs.),string,8
505,Other Prison data/Value_of_goods_produced_by_i...,after,Gross Value of Sale Proceeds / Earning (In Rs.),Float64,8
513,Prison_details_2015.csv,before,Contact No.,string,222
517,Prison_details_2015.csv,before,Location1,string,56
518,Prison_details_2015.csv,before,Location2,string,1210
519,Prison_details_2015.csv,before,Email ID,string,730


,file,source_record,column,action,old_value,new_value,reason
0,IPC_crime_inmates_convicted.csv,9023,Male 16-18 years,recover_from_agreeing_totals,NaN,0,Total 16-18 years; Total Male
1,Other Prison data/SLL_Crime_headwise_distribut...,8843,Female 18-30 years,recover_from_agreeing_totals,NaN,0,Total 18-30 years; Total Female


In [5]:
# inspect unresolved arithmetic differences instead of replacing source totals
totals = pd.read_csv(reports / "total_checks_failed.csv")
display(totals.loc[totals.stage.eq("after")])
flags = pd.read_csv(reports / "row_flags.csv")
print("Aggregation scopes:")
print(flags["scope"].value_counts().to_string())
print("Ambiguous key rows retained:", int(flags.ambiguous_key.sum()))

,file,stage,source_record,check,total_field,reported,calculated,difference
3,Other Prison data/Inmates_suffering_from_menta...,after,119,within_record,Total Under trial,49,46,3
4,Other Prison data/Inmates_suffering_from_menta...,after,316,within_record,Grand Total,160,260,-100
5,Other Prison data/Inmates_suffering_from_menta...,after,119,within_record,Total Female,6,0,6
7,Other Prison data/Number_of_women_prisoners_wi...,after,442,within_record,Total No. of children of Women Prisoners,0,2,-2


Aggregation scopes:
scope
detail                 148155
reported_total           6238
reported_subtotal        3159
directory                1397
separate_population        70
Ambiguous key rows retained: 143


In [6]:
# review outlier detection coverage; flags do not delete data
bounds = pd.read_csv(reports / "outlier_bounds.csv")
print("Fence groups checked:", int(bounds.status.eq("checked").sum()))
print("Zero-IQR groups skipped:", int(bounds.status.eq("zero_iqr_not_flagged").sum()))
print("Outlier cells retained:", int(bounds.flagged.sum()))
display(bounds.loc[bounds.flagged.gt(0)].head(10))

Fence groups checked: 1361
Zero-IQR groups skipped: 945
Outlier cells retained: 56018


,file,column,context,n,q1,q3,iqr,lower,upper,flagged,status
5,Age_group.csv,age_18_30,"{""category"": ""Foreigners"", ""type"": ""Convicts"",...",447,0.0,9.0,9.0,-13.5,22.5,49,checked
6,Age_group.csv,age_30_50,"{""category"": ""Foreigners"", ""type"": ""Convicts"",...",447,0.0,11.0,11.0,-16.5,27.5,69,checked
7,Age_group.csv,age_50_above,"{""category"": ""Foreigners"", ""type"": ""Convicts"",...",447,0.0,2.0,2.0,-3.0,5.0,51,checked
25,Age_group.csv,age_18_30,"{""category"": ""Foreigners"", ""type"": ""Under Tria...",447,0.0,1.0,1.0,-1.5,2.5,71,checked
26,Age_group.csv,age_30_50,"{""category"": ""Foreigners"", ""type"": ""Under Tria...",447,0.0,2.0,2.0,-3.0,5.0,57,checked
29,Age_group.csv,age_18_30,"{""category"": ""Foreigners"", ""type"": ""Under Tria...",447,0.0,21.0,21.0,-31.5,52.5,54,checked
30,Age_group.csv,age_30_50,"{""category"": ""Foreigners"", ""type"": ""Under Tria...",447,0.0,25.0,25.0,-37.5,62.5,52,checked
31,Age_group.csv,age_50_above,"{""category"": ""Foreigners"", ""type"": ""Under Tria...",447,0.0,2.0,2.0,-3.0,5.0,66,checked
33,Age_group.csv,age_18_30,"{""category"": ""Indians"", ""type"": ""Convicts"", ""g...",447,0.0,46.0,46.0,-69.0,115.0,23,checked
34,Age_group.csv,age_30_50,"{""category"": ""Indians"", ""type"": ""Convicts"", ""g...",447,1.0,94.0,93.0,-138.5,233.5,19,checked


In [7]:
# verify row accounting and raw-file hashes
assert (quality.rows_before == quality.rows_after + quality.duplicates_removed
        + quality.quarantined_rows).all()
manifest = json.loads((ROOT / "results/tables/dataset_inventory.json").read_text())
assert all(sha256(ROOT / "data/raw" / item["file"]) == item["sha256"] for item in manifest)
assert len(flags) == summary["rows_after"]
print("Row accounting, quality-flag coverage and all raw-file hashes verified.")

Row accounting, quality-flag coverage and all raw-file hashes verified.


### Interpretation and phase boundary

The tables are now parsed consistently, with reversible provenance, recorded repairs and quality flags. Remaining missing values, contradictory totals and repeated training/goods keys are unresolved source limitations, not evidence that their values are zero. High IQR values may reflect differences in state size, repeated years or sparse counts; they are not automatically errors.

The cleaned CSVs preserve reported totals as well as detail records. Always consult `row_flags.csv` and choose an explicit aggregation scope; do not add combined totals to their components or combine overlapping marginal demographic tables. The typed `load_cleaned` helper requires an explicit scope. Its merge only attaches quality metadata and does not integrate source datasets.

See [the cleaning report](../report/data_cleaning.md) for full rules, actual findings and limitations. The next section implements the approved Phase 2 selection work.

## Phase 2 Data Selection

We select native source fields and comparable observation keys before constructing any features. Four sources support a common state-year profile: jail population, population/capacity, age groups and education. We also define 27 source-specific descriptive views and defer ten administrative or temporally incompatible files. Files are not assumed comparable merely because their state/year strings match.

In [8]:
from src.preprocessing.selection import run_selection, load_descriptive_view
selection_reports = ROOT / "results/tables/selection"
selected_dir = ROOT / "data/processed/selected"

In [9]:
# validate keys, populations and denominators; save selected native counts
selection_summary = run_selection(ROOT)
print(json.dumps(selection_summary, indent=2))

{
  "files_reviewed": 37,
  "descriptive_candidate_files": 27,
  "deferred_files": 10,
  "primary_source_files": 4,
  "reference_state_years": 447,
  "profile_state_years": 437,
  "profile_excluded_state_years": 10,
  "profile_age_rows": 6992,
  "profile_education_rows": 5244,
  "planned_features": 5,
  "features_computed": false,
  "cleaned_hashes_unchanged": true
}


In [10]:
file_decisions = pd.read_csv(selection_reports / "file_decisions.csv")
display(file_decisions)
features = json.loads((selection_reports / "feature_contract.json").read_text())
display(pd.DataFrame(features)[["feature", "future_formula", "reason"]])

,file,primary_profile,descriptive_candidate,reason
0,Age_group.csv,True,True,Primary profile source; explicit common-cohort...
1,Caste.csv,False,True,Historical prisoner statistics retained for so...
2,Death_sentence.csv,False,True,Historical prisoner statistics retained for so...
3,Domicile.csv,False,True,Historical prisoner statistics retained for so...
4,Education.csv,True,True,Primary profile source; explicit common-cohort...
5,Education_facilities.csv,False,True,Historical prisoner statistics retained for so...
6,IPC_crime_inmates_convicted.csv,False,True,Historical prisoner statistics retained for so...
7,IPC_crime_inmates_under_trial.csv,False,True,Historical prisoner statistics retained for so...
8,Inmates_death.csv,False,True,Historical prisoner statistics retained for so...
9,Inmates_escapee.csv,False,True,Historical prisoner statistics retained for so...


,feature,future_formula,reason
0,occupancy_ratio,total_prisoners / total_capacity,"Capacity pressure, independent of absolute sta..."
1,undertrial_share,undertrial_prisoners / total_prisoners,Legal-status composition without complementary...
2,female_share,female_prisoners / total_prisoners,Aggregate gender composition; not an individua...
3,age_18_30_share,"sum age_18_30 over nationality, status and gen...",One age-composition measure; omit other comple...
4,illiterate_share,sum four status counts for education=Illiterat...,One education-composition measure with a verif...


The five feature names and formulas are a contract for Phase 3; the ratios are not computed here. State, year and source-record IDs are join/provenance fields rather than model inputs. We exclude complementary shares, raw population size, religion/caste labels and any future cluster label from the primary feature representation. The same eventual cohort and feature matrix will be used for Python/Spark comparisons.

In [11]:
display(pd.read_csv(selection_reports / "profile_exclusions.csv"))
margins = pd.read_csv(selection_reports / "marginal_compatibility.csv")
display(margins.loc[margins.file.isin(["Age_group.csv", "Education.csv"]) & ~margins.compatible])

,state,year,marginal_compatible,positive_denominators,reason
0,Karnataka,2003,False,True,Age/education completeness or population compa...
1,Lakshadweep,2004,True,False,Zero prisoner denominator; demographic proport...
2,Lakshadweep,2006,True,False,Zero prisoner denominator; demographic proport...
3,Lakshadweep,2012,True,False,Zero prisoner denominator; demographic proport...
4,Lakshadweep,2013,True,False,Zero prisoner denominator; demographic proport...
5,Maharashtra,2010,False,True,Age/education completeness or population compa...
6,Tamil Nadu,2002,False,True,Age/education completeness or population compa...
7,Tripura,2004,False,True,Age/education completeness or population compa...
8,Uttar Pradesh,2001,False,True,Age/education completeness or population compa...
9,West Bengal,2013,False,True,Age/education completeness or population compa...


,file,state,year,present,rows,expected_rows,grid_complete,missing_measure_cells,mismatch_fields,compatible
398,Age_group.csv,Tripura,2004,True,16,16,True,0,male_detenues;male_others,False
408,Age_group.csv,Uttar Pradesh,2001,True,16,16,True,0,male_detenues;male_others,False
446,Age_group.csv,West Bengal,2013,True,16,16,True,1,NaN,False
649,Education.csv,Karnataka,2003,False,0,12,False,0,NaN,False
708,Education.csv,Maharashtra,2010,True,10,12,False,0,male_convicts;male_under_trial;male_detenues;f...,False
830,Education.csv,Tamil Nadu,2002,True,8,12,False,0,male_convicts;male_under_trial;male_detenues;m...,False


We compare the full gender × legal-status population cells, not just the grand total. A missing age cell is not silently converted to zero by summation. Four valid zero-prisoner records are retained in the reference but excluded from ratio-based profiles because population shares would be undefined. Other incompatible records remain available in the unchanged cleaned sources.

In [12]:
contracts = json.loads((selection_reports / "join_contracts.json").read_text())
display(pd.DataFrame(contracts))
display(pd.read_csv(selection_reports / "native_population_disagreements.csv"))

,left,right,keys,cardinality,checked_rows,status,population_cells_compared,disagreeing_cells,use,reason
0,Jail wise population of prison inmates.csv,Other Prison data/Population_and_capacity_of_j...,"[state, year, jail_type]",one_to_one,4023.0,keys_valid_but_measures_disagree,12069.0,8.0,Do not integrate conflicting jail-type measure...,NaN
1,population totals,capacity totals,"[state, year]",one_to_one,447.0,verified,NaN,NaN,NaN,NaN
2,age/education native records,profile cohort keys,"[state, year]",many_to_one,NaN,verified_filter_only,NaN,NaN,NaN,NaN
3,Age_group.csv,Education.csv,"[state, year]",many_to_many,NaN,forbidden,NaN,NaN,NaN,Separate marginal distributions cannot establi...
4,2001–2013 tables,2015/2015–2016 tables,[state],NaN,NaN,forbidden,NaN,NaN,NaN,Dates and geographic boundaries are not aligned.


,state,year,jail_type,population_field,capacity_field,population_value,capacity_value
0,Punjab,2007,Central Jail,Grand_Total,Total Inmate population,11252,11245
1,Punjab,2007,District Jail,Grand_Total,Total Inmate population,2703,2710
2,Odisha,2005,District Jail,Total_Male,Male Inmate population,6678,6706
3,Odisha,2005,Sub Jail,Total_Male,Male Inmate population,5282,5254
4,Punjab,2007,Central Jail,Total_Male,Male Inmate population,10740,10733
5,Punjab,2007,District Jail,Total_Male,Male Inmate population,2594,2601
6,Odisha,2005,District Jail,Total_Female,Female Inmate population,186,158
7,Odisha,2005,Sub Jail,Total_Female,Female Inmate population,195,223


At jail-type level, the two sources have matching keys but disagree on some population values. Their state-year totals agree, so the primary profile uses only that verified common level. We never join native age and education rows on state/year to create fictitious joint demographics. Those marginal tables remain separate until independent state-year aggregation in Phase 3.

In [13]:
selected_manifest = json.loads((selection_reports / "selected_manifest.json").read_text())
display(pd.DataFrame([{ "file": item["file"], "rows": item["rows"],
                        "columns": len(item["dtypes"])} for item in selected_manifest]))
display(pd.read_csv(selection_reports / "source_view_coverage.csv"))

,file,rows,columns
0,population_capacity_reference.csv,447,16
1,profile_base_counts.csv,437,8
2,profile_age_counts.csv,6992,7
3,profile_education_counts.csv,5244,9


,file,rows,native_key_duplicate_rows,state_years,record_eligible_rows,group_quality_eligible_detail_rows,analysis_eligible_rows
0,Age_group.csv,7152,0,447,7151,7136,7104
1,Caste.csv,3568,0,446,3568,3568,3448
2,Death_sentence.csv,446,0,446,446,446,446
3,Domicile.csv,2676,0,446,2676,2676,2670
4,Education.csv,5346,0,446,5346,5346,5328
5,Education_facilities.csv,445,0,445,445,445,445
6,IPC_crime_inmates_convicted.csv,9385,0,447,9385,9385,9345
7,IPC_crime_inmates_under_trial.csv,9385,0,447,9385,9385,9345
8,Inmates_death.csv,4023,0,447,3576,3576,3576
9,Inmates_escapee.csv,1788,0,447,1788,1788,1788


In [14]:
# load source-specific descriptive fields using the explicit selection policy
education_view = load_descriptive_view(ROOT, "Education.csv")
assert education_view.analysis_eligible.all()
assert education_view.population_compatible.all()
print("Eligible native education records:", len(education_view))
print("These descriptive views have source-specific coverage; the primary profile uses the shared cohort.")

Eligible native education records: 5328
These descriptive views have source-specific coverage; the primary profile uses the shared cohort.


In [15]:
# verify provenance and the Phase 2 boundary
base = pd.read_csv(selected_dir / "profile_base_counts.csv")
assert not base.duplicated(["state", "year"]).any()
assert base.total_capacity.gt(0).all() and base.total_prisoners.gt(0).all()
assert not set(item["feature"] for item in features).intersection(base.columns)
for item in selected_manifest:
    assert sha256(selected_dir / item["file"]) == item["sha256"]
cleaned_manifest = json.loads((ROOT / "results/tables/cleaning/cleaned_manifest.json").read_text())
assert all(sha256(ROOT / "data/processed/cleaned" / item["file"]) == item["sha256"]
           for item in cleaned_manifest)
print("Selected outputs verified; cleaned inputs unchanged; no Phase 3 features computed.")

Selected outputs verified; cleaned inputs unchanged; no Phase 3 features computed.


### Phase 2 interpretation and limitations

The reference has 447 state-year observations and the primary shared cohort has 437. Ten observed state-years are excluded from profiles by explicit compatibility/denominator rules; eight other state-year keys are absent from the historical reference. We do not invent these records or make the panel balanced by filling zeros.

Selected outputs preserve native counts and provenance. Demographic source tables are separate marginal distributions, not individual prisoner records. Descriptive views exclude unsafe groups by default but cannot establish unique people across training programs, releases or overlapping categories. Monetary units and broader source definitions still require verification before related interpretations.

See [data_selection.md](../report/data_selection.md) for file/attribute decisions, join constraints, leakage safeguards and row exclusions. The next section implements the approved Phase 3 transformation work.

## Phase 3 Data Transformation

We construct five profile features for the selected 437 state-years, standardize them for K-Medoids, and discretize them into association-mining items. Each source is aggregated independently before a validated state-year join. The same cohort and saved representations are prepared for Python and Spark. No clustering, frequent-itemset mining, rules, benchmarking or analytical plots are run here.

In [16]:
import numpy as np
from src.preprocessing.transformation import (FEATURES, run_transformation, load_transformed,
                                               scale_profiles, discretize_profiles)
transformation_reports = ROOT / "results/tables/transformation"
transformed_dir = ROOT / "data/processed/transformed"

In [17]:
# build features, persist fitted parameters, and prepare all input formats
transformation_summary = run_transformation(ROOT)
print(json.dumps(transformation_summary, indent=2))

{
  "profiles": 437,
  "numeric_features": 5,
  "transactions": 437,
  "items_per_transaction": 5,
  "declared_items": 15,
  "observed_items": 15,
  "onehot_shape": [
    437,
    15
  ],
  "onehot_true_cells": 2185,
  "fit_rows": 437,
  "upstream_files_verified": 78,
  "raw_cleaned_selected_unchanged": true,
  "pyspark_executed": false,
  "mining_or_clustering_executed": false
}


### Aggregation and numeric representation

We sum age_18_30 over nationality, status and gender. We independently sum the four legal-status counts for illiterate prisoners over both genders. These state-year aggregates join the population/capacity base one-to-one. The four demographic shares use the same validated total prisoner denominator; occupancy uses total capacity. Counts remain integer and the resulting features use float64. State, year and source-record IDs are metadata rather than numeric predictors.

In [18]:
profiles = load_transformed(ROOT, "profile_features.csv")
counts = load_transformed(ROOT, "profile_counts.csv")
print("Profile matrix:", profiles[FEATURES].shape)
display(profiles.head(5))
assert len(counts) == len(profiles) == 437
assert np.isfinite(profiles[FEATURES].to_numpy()).all()
assert profiles[FEATURES[1:]].ge(0).all().all()
assert profiles[FEATURES[1:]].le(1).all().all()

Profile matrix: (437, 5)


,profile_id,state,year,occupancy_ratio,undertrial_share,female_share,age_18_30_share,illiterate_share
0,Andaman & Nicobar Islands|2001,Andaman & Nicobar Islands,2001,1.423792,0.433420,0.010444,0.281984,0.000000
1,Andaman & Nicobar Islands|2002,Andaman & Nicobar Islands,2002,1.230483,0.586103,0.024169,0.422961,0.000000
2,Andaman & Nicobar Islands|2003,Andaman & Nicobar Islands,2003,1.100372,0.689189,0.010135,0.611486,0.087838
3,Andaman & Nicobar Islands|2004,Andaman & Nicobar Islands,2004,0.715210,0.746606,0.013575,0.542986,0.022624
4,Andaman & Nicobar Islands|2005,Andaman & Nicobar Islands,2005,0.724919,0.705357,0.017857,0.410714,0.053571


### Scaling and fitted parameters

We apply z-score scaling with scikit-learn StandardScaler, using population variance (ddof=0). The fit scope is the entire selected cohort for descriptive unsupervised analysis; no held-out performance is claimed. Mean, variance, scale, feature order and fit-profile IDs are saved. A future holdout experiment must fit scaling and quantiles on training IDs only, then apply those saved parameters to other rows. Outliers are retained without clipping.

In [19]:
parameters = json.loads((transformation_reports / "parameters.json").read_text())
display(pd.read_csv(transformation_reports / "scaling_parameters.csv"))
scaled = load_transformed(ROOT, "clustering_features.csv")
np.testing.assert_allclose(scaled[FEATURES].mean(), 0, atol=1e-12, rtol=0)
np.testing.assert_allclose(scaled[FEATURES].std(ddof=0), 1, atol=1e-12, rtol=0)
np.testing.assert_array_equal(scale_profiles(profiles, parameters)[FEATURES], scaled[FEATURES])

,feature,mean,variance_ddof0,scale,fit_rows
0,occupancy_ratio,1.189940,0.367016,0.605818,437
1,undertrial_share,0.664764,0.020873,0.144475,437
2,female_share,0.038224,0.000389,0.019711,437
3,age_18_30_share,0.466552,0.013264,0.115171,437
4,illiterate_share,0.276484,0.016691,0.129192,437


### Discretization and categorical encoding

Occupancy uses fixed bands: x ≤ 1, 1 < x ≤ 1.5, and x > 1.5. The 150% cut is an explicit analytical assumption, not an official standard. Each demographic share uses the fitted 1/3 and 2/3 quantiles with linear interpolation. Equality belongs to the lower interval. Ties are not split by rank, so bin sizes need not be exactly equal; indistinguishable cut points would fail rather than silently force categories. Tertile labels describe this cohort, not absolute social risk.

In [20]:
display(pd.read_csv(transformation_reports / "discretization_parameters.csv"))
discrete = load_transformed(ROOT, "discretized_profiles.csv")
replayed = discretize_profiles(profiles, parameters)
for feature in FEATURES:
    assert replayed[feature + "_bin"].tolist() == discrete[feature + "_bin"].tolist()
display(discrete.head(5))

,feature,method,first_cut,second_cut,labels,comparison
0,occupancy_ratio,fixed occupancy bands,1.000000,1.500000,within_capacity;over_100_to_150pct;over_150pct,lower intervals include the right boundary
1,undertrial_share,cohort tertiles,0.612654,0.705518,lower_tertile;middle_tertile;upper_tertile,lower intervals include the right boundary
2,female_share,cohort tertiles,0.030170,0.043353,lower_tertile;middle_tertile;upper_tertile,lower intervals include the right boundary
3,age_18_30_share,cohort tertiles,0.419582,0.483002,lower_tertile;middle_tertile;upper_tertile,lower intervals include the right boundary
4,illiterate_share,cohort tertiles,0.230263,0.338157,lower_tertile;middle_tertile;upper_tertile,lower intervals include the right boundary


,profile_id,state,year,occupancy_ratio_bin,undertrial_share_bin,female_share_bin,age_18_30_share_bin,illiterate_share_bin
0,Andaman & Nicobar Islands|2001,Andaman & Nicobar Islands,2001,over_100_to_150pct,lower_tertile,lower_tertile,lower_tertile,lower_tertile
1,Andaman & Nicobar Islands|2002,Andaman & Nicobar Islands,2002,over_100_to_150pct,lower_tertile,lower_tertile,middle_tertile,lower_tertile
2,Andaman & Nicobar Islands|2003,Andaman & Nicobar Islands,2003,over_100_to_150pct,middle_tertile,lower_tertile,upper_tertile,lower_tertile
3,Andaman & Nicobar Islands|2004,Andaman & Nicobar Islands,2004,within_capacity,upper_tertile,lower_tertile,upper_tertile,lower_tertile
4,Andaman & Nicobar Islands|2005,Andaman & Nicobar Islands,2005,within_capacity,middle_tertile,lower_tertile,lower_tertile,lower_tertile


In [21]:
# verify one categorical item per profile feature
transactions = load_transformed(ROOT, "transactions.jsonl")
onehot = load_transformed(ROOT, "transactions_onehot.csv")
items = [c for c in onehot if c != "profile_id"]
assert onehot[items].sum(axis=1).eq(5).all()
assert len(transactions) == len(profiles)
for i, transaction in enumerate(transactions):
    assert transaction["items"] == sorted(c for c in items if onehot.at[i, c])
    assert {s.split("=")[0] for s in transaction["items"]} == set(FEATURES)
print("One-hot matrix:", onehot[items].shape)
print("True cells:", int(onehot[items].sum().sum()))
print("First transaction:", transactions[0]["items"])

One-hot matrix: (437, 15)
True cells: 2185
First transaction: ['age_18_30_share=lower_tertile', 'female_share=lower_tertile', 'illiterate_share=lower_tertile', 'occupancy_ratio=over_100_to_150pct', 'undertrial_share=lower_tertile']


Repeated item patterns across different state-years are preserved. Support will later describe the fraction of the 437 state-year transactions, not the fraction of prisoners. We neither expand aggregate counts into people nor use state/year/provenance as items. The Boolean matrix encodes nominal membership only; no ordinal category codes are used for clustering.

In [22]:
# verify the exported Spark vectors without starting Spark
spark_rows = load_transformed(ROOT, "spark_profiles.jsonl")
assert [r["profile_id"] for r in spark_rows] == scaled.profile_id.tolist()
np.testing.assert_array_equal(np.array([r["features"] for r in spark_rows]), scaled[FEATURES].to_numpy())
contract = json.loads((transformation_reports / "spark_input_contract.json").read_text())
assert contract["feature_order"] == FEATURES
print("JSON vector order and values exactly match Python inputs.")
print("Spark runtime execution remains Phase 9/10 work.")

JSON vector order and values exactly match Python inputs.
Spark runtime execution remains Phase 9/10 work.


In [23]:
# check every output manifest and all upstream file hashes
manifest3 = json.loads((transformation_reports / "transformed_manifest.json").read_text())
for item in manifest3:
    assert sha256(transformed_dir / item["file"]) == item["sha256"]
inputs3 = json.loads((transformation_reports / "input_provenance.json").read_text())
assert all(sha256(ROOT / item["file"]) == item["sha256"] for item in inputs3)
display(pd.DataFrame([{ "file": r["file"], "rows": r["rows"], "format": r["format"] } for r in manifest3]))
print("All transformed outputs and 78 upstream files verified.")

,file,rows,format
0,profile_counts.csv,437,csv
1,profile_features.csv,437,csv
2,clustering_features.csv,437,csv
3,discretized_profiles.csv,437,csv
4,transactions_onehot.csv,437,csv
5,transactions.jsonl,437,jsonl
6,spark_profiles.jsonl,437,jsonl


All transformed outputs and 78 upstream files verified.


### Phase 3 interpretation and boundary

The numeric and categorical inputs are ready for later analysis and algorithms. Scaling parameters and quantile cut points are transformation parameters, not a completed descriptive analysis. StandardScaler can be influenced by extreme values; binning loses within-bin information; and full-cohort quantiles depend on the chosen cohort. Repeated years are correlated observations. Preserve these limitations when interpreting later clusters and rules.

See [data_transformation.md](../report/data_transformation.md) for formulas, exact parameters, output formats, lineage and fit-scope safeguards. Phase 3 stops here. Phase 4 Descriptive Analytics requires separate approval.